## Imports and Configs

In [1]:
##### Imports & Installations #####

import time
import pandas as pd
import torch
from IPython.display import display

from src.colpali.ingestion import ColPaliEncoder
from src.colpali.retrieval import ColPaliQdrantRetriever

In [3]:
##### Congigurations & Benchmark Queries #####

MODEL_NAME = "vidore/colpali-v1.3"
DEVICE = "mps"
QDRANT_PATH = ( "../../data/processed/colpali/qdrant")
COLLECTION_NAME = "colpali_pages"
TOP_K = 5

TOP_K = 5
N_WARMUP = 5
N_MEASURED = 20

print(f"Warm-up queries : {N_WARMUP}")
print(f"Measured queries: {N_MEASURED}")
print(f"Top-k           : {TOP_K}")

benchmark_queries = [
    "What is the main idea of LoRA?",
    "Why does LoRA reduce the number of trainable parameters?",
    "How are the low-rank matrices A and B used in LoRA?",
    "What is the objective function used by the proposed method?",
    "What are the main experimental results reported in the paper?",
    "How does LoRA compare with full fine-tuning?",
    "Why are the original model parameters frozen during LoRA training?",
    "What rank is used for the low-rank adaptation matrices?",
    "What does the main architecture of the proposed method look like?",
    "What are the advantages of LoRA during deployment?",
    "How does LoRA affect GPU memory consumption?",
    
    "What models are evaluated in the experiments?",
    "What datasets are used for evaluating the method?",
    "How does LoRA achieve parameter efficiency?",
    "What happens during the forward pass in LoRA?",
    "Why is the update matrix represented as BA?",
    "What are the limitations of the proposed approach?",
    "What does the paper conclude about low-rank adaptation?",
    "How does LoRA compare with other parameter-efficient methods?",
    "What are the key findings of the paper?"]

assert len(benchmark_queries) == N_MEASURED
print(f"Prepared {len(benchmark_queries)} benchmark queries.")

Warm-up queries : 5
Measured queries: 20
Top-k           : 5
Prepared 20 benchmark queries.


In [4]:
##### Loading ColPali Encoder and Retriever #####

print("Loading ColPali...")
encoder = ColPaliEncoder(model_name=MODEL_NAME,device=DEVICE)
print("ColPali loaded...")
model_device = next(encoder.model.parameters()).device

retriever = ColPaliQdrantRetriever(encoder=encoder,collection_name=COLLECTION_NAME,qdrant_path=QDRANT_PATH)
print("Retriever initialized...")

Loading ColPali...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/605 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

ColPali loaded...
Retriever initialized...


## Warm-Up

In [5]:
warmup_queries = [
    "What is LoRA?",
    "Explain the main idea of the paper.",
    "What is the proposed method?",
    "What are the experimental results?",
    "What is the main contribution?"
]

print("=" * 70)
print("WARM-UP")
print("=" * 70)

warmup_times = []

for i, query in enumerate(warmup_queries, start=1):
    _, timing = retriever.retrieve_with_timing(
        query=query,
        top_k=TOP_K,
    )

    warmup_times.append(timing["total_ms"])

    print(
        f"Warm-up {i}: "
        f"{timing['total_ms']:.2f} ms "
        f"(encoding={timing['encoding_ms']:.2f} ms, "
        f"qdrant={timing['qdrant_ms']:.2f} ms)"
    )

print("\nWarm-up complete.")

print(
    f"Mean warm-up latency: "
    f"{sum(warmup_times) / len(warmup_times):.2f} ms"
)

WARM-UP
Warm-up 1: 23911.46 ms (encoding=15739.71 ms, qdrant=8171.72 ms)
Warm-up 2: 7898.29 ms (encoding=6816.75 ms, qdrant=1081.52 ms)
Warm-up 3: 2784.63 ms (encoding=1356.09 ms, qdrant=1428.53 ms)
Warm-up 4: 1786.13 ms (encoding=873.66 ms, qdrant=912.46 ms)
Warm-up 5: 2531.42 ms (encoding=1116.36 ms, qdrant=1415.04 ms)

Warm-up complete.
Mean warm-up latency: 7782.39 ms


### MPS State Check

In [7]:
if torch.backends.mps.is_available():
    print(
        f"Current allocated MPS memory: "
        f"{torch.mps.current_allocated_memory() / (1024 ** 2):.2f} MB"
    )

    print(
        f"Driver allocated MPS memory: "
        f"{torch.mps.driver_allocated_memory() / (1024 ** 2):.2f} MB"
    )

else:
    print("MPS is not available.")

Current allocated MPS memory: 0.01 MB
Driver allocated MPS memory: 5670.64 MB


## Benchmarking Queries

In [ ]:
####### Steady-State Benchmarking #######

benchmark_results = []

print("=" * 70)
print("STEADY-STATE RETRIEVAL BENCHMARK")
print("=" * 70)

for i, query in enumerate(benchmark_queries, start=1):

    _, timing = retriever.retrieve_with_timing(query=query,top_k=TOP_K,)
    benchmark_results.append(
        {
            "run": i,
            "query": query,
            "query_chars": len(query),
            "encoding_ms": timing["encoding_ms"],
            "qdrant_ms": timing["qdrant_ms"],
            "total_ms": timing["total_ms"],
        }
    )

    print(
        f"[{i:02d}/{N_MEASURED}] "
        f"encoding={timing['encoding_ms']:8.2f} ms | "
        f"qdrant={timing['qdrant_ms']:8.2f} ms | "
        f"total={timing['total_ms']:8.2f} ms"
    )


STEADY-STATE RETRIEVAL BENCHMARK
[01/20] encoding=  828.37 ms | qdrant= 1908.47 ms | total= 2736.86 ms
[02/20] encoding=  952.45 ms | qdrant= 1547.14 ms | total= 2499.60 ms
[03/20] encoding=  813.36 ms | qdrant=  712.54 ms | total= 1525.91 ms
[04/20] encoding=  185.46 ms | qdrant=  223.01 ms | total=  408.48 ms
[05/20] encoding=   76.51 ms | qdrant=  180.73 ms | total=  257.24 ms
[06/20] encoding=   74.37 ms | qdrant=  139.13 ms | total=  213.50 ms
[07/20] encoding=   78.30 ms | qdrant=  151.22 ms | total=  229.53 ms
[08/20] encoding=   70.12 ms | qdrant=  139.78 ms | total=  209.90 ms
[09/20] encoding=   78.16 ms | qdrant=  160.19 ms | total=  238.35 ms
[10/20] encoding=   92.78 ms | qdrant=  786.65 ms | total=  879.45 ms
[11/20] encoding=  100.74 ms | qdrant=  189.65 ms | total=  290.40 ms
[12/20] encoding=   69.84 ms | qdrant=  149.84 ms | total=  219.69 ms
[13/20] encoding=   71.56 ms | qdrant=  196.80 ms | total=  268.37 ms
[14/20] encoding=   72.35 ms | qdrant=  245.84 ms | total

In [9]:
benchmark_df = pd.DataFrame(benchmark_results)
display(benchmark_df)

,run,query,query_chars,encoding_ms,qdrant_ms,total_ms
0,1,What is the main idea of LoRA?,30,828.368292,1908.469250,2736.859958
1,2,Why does LoRA reduce the number of trainable p...,56,952.450958,1547.139167,2499.597625
2,3,How are the low-rank matrices A and B used in ...,51,813.364209,712.540000,1525.906750
3,4,What is the objective function used by the pro...,59,185.456750,223.014958,408.475875
4,5,What are the main experimental results reporte...,61,76.507292,180.734583,257.243750
5,6,How does LoRA compare with full fine-tuning?,44,74.371875,139.131041,213.504833
6,7,Why are the original model parameters frozen d...,66,78.302500,151.223917,229.528250
7,8,What rank is used for the low-rank adaptation ...,55,70.122042,139.776417,209.900625
8,9,What does the main architecture of the propose...,65,78.155334,160.192750,238.351167
9,10,What are the advantages of LoRA during deploym...,50,92.779375,786.652833,879.449333


In [10]:
summary = pd.DataFrame({
    "metric": [
        "encoding_ms",
        "qdrant_ms",
        "total_ms",
    ],

    "mean_ms": [
        benchmark_df["encoding_ms"].mean(),
        benchmark_df["qdrant_ms"].mean(),
        benchmark_df["total_ms"].mean(),
    ],

    "median_ms": [
        benchmark_df["encoding_ms"].median(),
        benchmark_df["qdrant_ms"].median(),
        benchmark_df["total_ms"].median(),
    ],

    "p95_ms": [
        benchmark_df["encoding_ms"].quantile(0.95),
        benchmark_df["qdrant_ms"].quantile(0.95),
        benchmark_df["total_ms"].quantile(0.95),
    ],

    "std_ms": [
        benchmark_df["encoding_ms"].std(),
        benchmark_df["qdrant_ms"].std(),
        benchmark_df["total_ms"].std(),
    ],
})

display(summary)

,metric,mean_ms,median_ms,p95_ms,std_ms
0,encoding_ms,200.501921,76.119354,834.572425,288.457537
1,qdrant_ms,390.807342,185.193937,1565.205671,494.035523
2,total_ms,591.313925,262.806396,2511.460742,760.271865


In [11]:
slowest = (benchmark_df.sort_values("total_ms", ascending=False).head(5))
display(slowest)

,run,query,query_chars,encoding_ms,qdrant_ms,total_ms
0,1,What is the main idea of LoRA?,30,828.368292,1908.469250,2736.859958
1,2,Why does LoRA reduce the number of trainable p...,56,952.450958,1547.139167,2499.597625
2,3,How are the low-rank matrices A and B used in ...,51,813.364209,712.540000,1525.906750
9,10,What are the advantages of LoRA during deploym...,50,92.779375,786.652833,879.449333
3,4,What is the objective function used by the pro...,59,185.456750,223.014958,408.475875


### Auxilliary analysis after outlier removal

In [17]:
steady_df = (benchmark_df.sort_values("total_ms").iloc[:-3])
print(
    f"All runs median: "
    f"{benchmark_df['total_ms'].median():.2f} ms"
)

print(
    f"Without slowest run median: "
    f"{steady_df['total_ms'].median():.2f} ms"
)

print(
    f"All runs mean: "
    f"{benchmark_df['total_ms'].mean():.2f} ms"
)

print(
    f"Without slowest run mean: "
    f"{steady_df['total_ms'].mean():.2f} ms"
)

All runs median: 262.81 ms
Without slowest run median: 252.90 ms
All runs mean: 591.31 ms
Without slowest run mean: 297.88 ms


### Saving Results

In [14]:
from pathlib import Path

RESULTS_DIR = Path("../../results/colpali")
RESULTS_DIR.mkdir(parents=True,exist_ok=True,)

benchmark_df.to_csv(RESULTS_DIR / "retrieval_latency_raw.csv",index=False,)
summary.to_csv(RESULTS_DIR / "retrieval_latency_summary.csv",index=False,)

print(f"Results saved to: {RESULTS_DIR.resolve()}")

Results saved to: /Users/arjunmallick/4. Generative AI (Interview)/scientific-literature-rag/results/colpali


## Final Report

In [15]:
print("=" * 70)
print("COLPALI STEADY-STATE RETRIEVAL BENCHMARK")
print("=" * 70)

print(f"Number of warm-up queries : {N_WARMUP}")
print(f"Number of measured queries: {N_MEASURED}")
print(f"Top-k                     : {TOP_K}")

print("\nLatency:")
print(
    f"Encoding  - "
    f"mean: {benchmark_df['encoding_ms'].mean():.2f} ms | "
    f"median: {benchmark_df['encoding_ms'].median():.2f} ms | "
    f"P95: {benchmark_df['encoding_ms'].quantile(.95):.2f} ms"
)

print(
    f"Qdrant    - "
    f"mean: {benchmark_df['qdrant_ms'].mean():.2f} ms | "
    f"median: {benchmark_df['qdrant_ms'].median():.2f} ms | "
    f"P95: {benchmark_df['qdrant_ms'].quantile(.95):.2f} ms"
)

print(
    f"Total     - "
    f"mean: {benchmark_df['total_ms'].mean():.2f} ms | "
    f"median: {benchmark_df['total_ms'].median():.2f} ms | "
    f"P95: {benchmark_df['total_ms'].quantile(.95):.2f} ms"
)

COLPALI STEADY-STATE RETRIEVAL BENCHMARK
Number of warm-up queries : 5
Number of measured queries: 20
Top-k                     : 5

Latency:
Encoding  - mean: 200.50 ms | median: 76.12 ms | P95: 834.57 ms
Qdrant    - mean: 390.81 ms | median: 185.19 ms | P95: 1565.21 ms
Total     - mean: 591.31 ms | median: 262.81 ms | P95: 2511.46 ms
